In [1]:
import pybaseball as pb
import pandas as pd
import numpy as np

pb.cache.enable()

/Users/arjangunsi/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pb.statcast(start_dt="2023-06-01", end_dt="2023-06-30")
print(df.shape)
print(df.columns.tolist())

This is a large query, it may take a moment to complete


  0%|          | 0/30 [00:00<?, ?it/s]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explic

(115544, 118)
['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'babip_value', 'iso_value', '

In [3]:
# Check what outcome events look like
print(df['events'].value_counts().head(20))
print()

# Isolate home runs
hrs = df[df['events'] == 'home_run']
fly_balls = df[df['bb_type'] == 'fly_ball']

print(f"Total pitches: {len(df)}")
print(f"Home runs: {len(hrs)}")
print(f"Fly balls: {len(fly_balls)}")
print(f"HR rate among fly balls: {len(hrs)/len(fly_balls)*100:.1f}%")

events
field_out                    11716
strikeout                     6705
single                        4188
walk                          2413
double                        1332
home_run                       913
force_out                      550
grounded_into_double_play      550
hit_by_pitch                   327
sac_fly                        211
field_error                    156
triple                         116
fielders_choice                 71
sac_bunt                        65
double_play                     65
intent_walk                     59
truncated_pa                    50
fielders_choice_out             39
catcher_interf                  16
strikeout_double_play           14
Name: count, dtype: int64

Total pitches: 115544
Home runs: 913
Fly balls: 5239
HR rate among fly balls: 17.4%


In [4]:
# Look at the profile of a home run vs non-HR fly ball
fb_df = df[df['bb_type'] == 'fly_ball'].copy()
fb_df['is_hr'] = (fb_df['events'] == 'home_run').astype(int)

# Compare avg launch speed and angle between HRs and non-HRs
comparison = fb_df.groupby('is_hr')[['launch_speed', 'launch_angle', 'hit_distance_sc']].mean()
print(comparison)
print()

# What pitch types get hit for HRs most?
print(df[df['events'] == 'home_run']['pitch_name'].value_counts())
print()

# Handedness breakdown
print("HR by batter stance:")
print(hrs['stand'].value_counts())
print()
print("HR by pitcher throwing arm:")
print(hrs['p_throws'].value_counts())

       launch_speed  launch_angle  hit_distance_sc
is_hr                                             
0         90.029615     38.368027       301.416288
1        104.239445     29.021713       402.657385

pitch_name
4-Seam Fastball    353
Slider             145
Sinker              94
Changeup            89
Cutter              87
Curveball           53
Sweeper             53
Split-Finger        20
Knuckle Curve       13
Other                3
Slurve               3
Name: count, dtype: int64

HR by batter stance:
stand
R    544
L    369
Name: count, dtype: int64

HR by pitcher throwing arm:
p_throws
R    707
L    206
Name: count, dtype: int64


In [5]:
key_features = [
    'launch_speed', 'launch_angle', 'hit_distance_sc',
    'release_speed', 'release_spin_rate', 'p_throws', 
    'stand', 'pitch_type', 'bb_type', 'home_team',
    'pitcher_days_since_prev_game', 'bat_speed', 'swing_length'
]

print("Null counts in key features:")
print(fb_df[key_features].isnull().sum())
print()
print(f"Total fly ball rows: {len(fb_df)}")

Null counts in key features:
launch_speed                       0
launch_angle                       0
hit_distance_sc                   17
release_speed                      0
release_spin_rate                 66
p_throws                           0
stand                              0
pitch_type                         0
bb_type                            0
home_team                          0
pitcher_days_since_prev_game     139
bat_speed                       5239
swing_length                    5239
dtype: int64

Total fly ball rows: 5239


In [6]:
print("Pulling 2022...")
df_2022 = pb.statcast(start_dt="2022-04-07", end_dt="2022-10-05")

print("Pulling 2023...")
df_2023 = pb.statcast(start_dt="2023-04-01", end_dt="2023-10-01")

print("Pulling 2024...")
df_2024 = pb.statcast(start_dt="2024-03-20", end_dt="2024-09-29")

print("Pulling 2025...")
df_2025 = pb.statcast(start_dt="2025-03-27", end_dt="2025-09-28")

# Combine
df_all = pd.concat([df_2022, df_2023, df_2024, df_2025], ignore_index=True)
print(f"\nTotal rows: {len(df_all)}")
print(f"Shape: {df_all.shape}")

Pulling 2022...
This is a large query, it may take a moment to complete


  0%|          | 0/182 [00:00<?, ?it/s]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 1/182 [00:04<12:45,  4.23s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 2/182 [00:04<05:24,  1.80s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will 

Pulling 2023...
This is a large query, it may take a moment to complete


  0%|          | 0/184 [00:00<?, ?it/s]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 1/184 [00:03<09:30,  3.12s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 2/184 [00:03<05:21,  1.76s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will 

Pulling 2024...
This is a large query, it may take a moment to complete


  1%|          | 1/194 [00:01<04:04,  1.26s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 2/194 [00:03<06:36,  2.06s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  2%|▏         | 3/194 [00:04<04:24,  1.38s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated a

Pulling 2025...
This is a large query, it may take a moment to complete


  0%|          | 0/186 [00:00<?, ?it/s]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 1/186 [00:01<04:31,  1.47s/it]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  1%|          | 2/186 [00:01<02:43,  1.13it/s]/Users/arjangunsi/.venv/lib/python3.9/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will 


Total rows: 2868714
Shape: (2868714, 118)


In [8]:
import os

save_path = os.path.join(os.path.expanduser("~"), "hr-projector", "data", "raw", "statcast_2022_2025.parquet")
df_all.to_parquet(save_path, index=False)
print(f"Saved to {save_path}")

Saved to /Users/arjangunsi/hr-projector/data/raw/statcast_2022_2025.parquet
